In [ ]:
import pymupdf  
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import redis
from redis.commands.search.field import (
    VectorField,
    TagField,
    TextField
)
from redis.commands.search.index_definition import (
    IndexDefinition,
    IndexType
)

In [2]:
PDF_FILE_PATH = r"C:\Users\noah.ross\downloads\Strategy_Flow\Army_Equipment_Guide.pdf"
REDIS_HOST = "localhost"
REDIS_PORT = 6379
INDEX_NAME = "army_equipment_idx"
PREFIX = "doc:"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

In [3]:
print("--- Step 1: Extracting Text from PDF ---")
full_text = ""
try:
    with pymupdf.open(PDF_FILE_PATH) as doc:
        for page in doc:
            full_text += page.get_text()
    print(f"Successfully extracted text from '{PDF_FILE_PATH}'.")
except Exception as e:
    print(f"Error reading PDF: {e}")
    exit()


--- Step 1: Extracting Text from PDF ---
Successfully extracted text from 'C:\Users\noah.ross\downloads\Strategy_Flow\Army_Equipment_Guide.pdf'.


In [4]:
print("\n--- Step 2: Chunking text into reports ---")
document_to_embed = []
metadata = []
ids = []
report_chunks = re.split(r"(?=WEG Location:)", full_text)

for i, chunk in enumerate(report_chunks):
    if chunk.strip() == "":
        continue
    
    report_title = f"Report {i+1}"
        
    lines = chunk.strip().splitlines()
    cleaned_lines = [line for line in lines if "For Training Use Only" not in line and "Exported (UTC)" not in line]
    processed_text = " ".join(cleaned_lines).strip()

    if len(processed_text) > 50:
        document_to_embed.append(processed_text)
        metadata.append({'source_file': PDF_FILE_PATH, 'report_title': report_title})
        ids.append(f"{PDF_FILE_PATH}_report_{i+1}")

print(f"Successfully created {len(document_to_embed)} chunks (one per report).")


--- Step 2: Chunking text into reports ---
Successfully created 3601 chunks (one per report).


In [ ]:
print("\n--- Step 3: Creating embeddings ---")
model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = model.encode(document_to_embed, show_progress_bar=True)
print(f"Embeddings created with shape: {embeddings.shape}")

In [ ]:
print("\n--- Step 4: Loading data into Redis ---")
try:
    r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT)
    r.ping()
    print("Successfully connected to Redis.")

    VECTOR_DIM = embeddings.shape[1]
    schema = (
        TextField("content"),
        TagField("report_title"),
        VectorField("vector", "HNSW", {"TYPE": "FLOAT32", "DIM": VECTOR_DIM, "DISTANCE_METRIC": "COSINE"}),
    )

    definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
    try:
        r.ft(INDEX_NAME).dropindex(delete_documents=True)
        print("Deleted existing index.")
    except redis.exceptions.ResponseError:
        print("Index did not exist, creating new one.")
    r.ft(INDEX_NAME).create_index(fields=schema, definition=definition)

    pipeline = r.pipeline()
    for i, doc_text in enumerate(document_to_embed):
        vector_bytes = np.array(embeddings[i], dtype=np.float32).tobytes()
        item_data = {
            "content": doc_text,
            "report_title": metadata[i]["report_title"],
            "vector": vector_bytes
        }
        pipeline.hset(f"{PREFIX}{ids[i]}", mapping=item_data)
    pipeline.execute()
    
    print(f"\n✅ Data loaded successfully! The Redis index '{INDEX_NAME}' now contains {len(ids)} documents.")

except Exception as e:
    print(f"\nAn error occurred during the Redis step: {e}")

c:\Users\noah.ross\Downloads\Strategy_Flow\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Step 1: Extracting Text from PDF ---
Successfully extracted text from 'C:\Users\noah.ross\downloads\Strategy_Flow\Army_Equipment_Guide.pdf'.

--- Step 2: Chunking text into reports ---
Successfully created 3601 chunks (one per report).

--- Step 3: Creating embeddings ---


Batches: 100%|██████████| 113/113 [01:00<00:00,  1.86it/s]


Embeddings created with shape: (3601, 384)

--- Step 4: Loading data into Redis ---
Successfully connected to Redis.
Index did not exist, creating new one.

✅ Data loaded successfully! The Redis index 'army_equipment_idx' now contains 3601 documents.
